# Proximity Analysis

Goal: Understand how predicted graphs compare to ground truth graphs with respect to proximity of nodes.
This affects the node matching process:
1) If no GT node is within a proximity (=vessel radius of node) of a predicted node then there will be no matching.
2) If multiple GT nodes lie within a proximity we need a rule how to select the "best" proximal GT node for matching.

In this notebook we create a table that shows the following for every source (i.e. prediction) graph's node v:
- node index
- node radius
- number of GT candidate nodes within that radius around v
- Boolean if v has parent
- Boolean if all candidates have parents
- The distances from v to its candidates

In [13]:
import networkx as nx
import numpy as np
import pandas as pd

from project.utils import read_graph_from_file

def matching_analysis(source, target):
    source_positions = nx.get_node_attributes(source, 'coord')
    source_radii = nx.get_node_attributes(source, 'radius')
    target_positions = nx.get_node_attributes(target, 'coord')
    if not source_positions or not target_positions:
        raise ValueError("Both source and target graphs must have 'coord' attributes for nodes.")

    data = {
        "node" : [],
        "radius" : [],
        "nr_candidates" : [],
        "v_has_parent" : [],
        "candidates_have_parent" : [],
        "dist_to_cands" : [],
        "parents_dist": [],
    }
    for source_node, source_pos in source_positions.items():
        radius = source_radii[source_node]
        candidates = {t_node : t_pos for t_node, t_pos in target_positions.items() if np.linalg.norm(np.array(source_pos) - np.array(t_pos)) < radius}

        def has_parent(graph, node):
            return len(list(graph.predecessors(node))) > 0
        
        cand_parents = [has_parent(target, t_node) for t_node in candidates.keys()] 
        dist_to_cands = [float(np.linalg.norm(np.array(source_pos) - np.array(t_pos))) for t_pos in candidates.values()]

        data["node"].append(source_node)
        data["radius"].append(radius)
        data["nr_candidates"].append(len(candidates))
        data["v_has_parent"].append(has_parent(source, source_node))
        data["candidates_have_parent"].append(all(cand_parents))
        data["dist_to_cands"].append(dist_to_cands)
    return pd.DataFrame(data)

The following block returns a table of the relevant information per predicted node.

In [7]:
gt = read_graph_from_file("tests/toygraph_GT.swc")
pred = read_graph_from_file("tests/toygraph_predicted.swc")

match_pd = matching_analysis(pred, gt)
match_pd


,node,radius,nr_candidates,v_has_parent,candidates_have_parent,dist_to_cands,parents_dist
0,1,6.200549,1,False,False,[0.42291796218661076],None
1,2,6.200549,1,True,True,[0.4451159843290837],None
2,3,6.195369,1,True,True,[0.6727754917320966],None
3,4,1.170284,1,True,True,[0.42275551082775226],None
4,5,1.955743,1,True,True,[0.3878164708015262],None
...,...,...,...,...,...,...,...
96,97,1.127999,1,True,True,[0.5123863572900917],None
97,98,1.127999,1,True,True,[0.2690986512674763],None
98,99,0.952582,1,True,True,[0.4769524214615983],None
99,100,0.952582,1,True,True,[0.5796724160478819],None


## Observation
Let's see how many predicted nodes have more than one GT candidates, or zero candidates

In [14]:
match_pd[match_pd["nr_candidates"]>1]

,node,radius,nr_candidates,v_has_parent,candidates_have_parent,dist_to_cands,parents_dist


In [15]:
match_pd[match_pd["nr_candidates"]==0]

,node,radius,nr_candidates,v_has_parent,candidates_have_parent,dist_to_cands,parents_dist


In the above example, all predicted nodes have exactly one GT candidate within proximity.